# Knowledge Extraction from Atomically Resolved Images: FeSeTe

**IMC-21: Artificial Intelligence Methods for Microscopy Analysis and Knowledge Extraction**

*Workshop notebook based on Vlcek, Maksov, Pan, Vasudevan, and Kalinin, "Knowledge Extraction from Atomically Resolved Images".*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pycroscopy/IMC21-Workshop/blob/main/notebooks/13_00_Knowledge_Extraction_FeSeTe.ipynb)

---

## Introduction

This notebook recreates the central idea of the paper in a compact, runnable form. We simulate an atomically resolved STM image of the chalcogen surface of FeSe$_x$Te$_{1-x}$, recover the Se/Te arrangement, and infer an effective interaction energy from local configuration statistics.

The paper uses a physics codebook: all symmetry-distinct nearest-neighbor surface configurations. Here we compare two routes:

1. **Physics codebook**: assume atom identities are known and count 12 local configurations.
2. **Windowed k-means codebook**: use `pycroscopy.image.ImageWindowing` to extract local image windows, cluster them with k-means, then build the same configuration histogram from the learned labels.

The payoff is that a single atomically resolved image can constrain a generative model, not just produce a pretty segmentation.

<div style="max-width: 80%; border-left: 6px solid #2e7d32; border-top: 1px solid #e0e0e0; border-right: 1px solid #e0e0e0; border-bottom: 1px solid #e0e0e0; border-radius: 4px; padding: 0; margin-bottom: 20px; box-shadow: 0 4px 8px rgba(0,0,0,0.1), 0 1px 3px rgba(0,0,0,0.08); background-color: #ffffff;">
  <div style="background-color: transparent; color: #1b5e20; padding: 10px 15px; font-weight: bold; border-bottom: 1px solid #e0e0e0; display: flex; align-items: center; gap: 8px;">
    <span>Target</span> Learning Goals
  </div>
  <div style="padding: 15px; background-color: transparent; color: #333333;">
  <ul style="margin: 0; padding-left: 20px;">
      <li style="margin-bottom: 8px;">Connect atomically resolved images to local structural descriptors.</li>
      <li style="margin-bottom: 8px;">Use `pycroscopy` windowing to turn an image into a matrix of local observations.</li>
      <li style="margin-bottom: 8px;">Compare a physics-defined codebook with a data-driven k-means codebook.</li>
      <li style="margin-bottom: 0;">Estimate how image noise propagates into uncertainty on interaction energies.</li>
    </ul>
  </div>
</div>

## 0. Setup

The imports are Colab-safe. If `pycroscopy` is not already available, the first cell installs the minimum packages used here. The notebook uses a small 2D surface model for speed; the paper's full model includes interactions with adjacent chalcogen layers, whose parameters are weakly constrained by a single surface image.

In [ ]:
# --- Setup: install packages (Colab-safe). Skip if you already have them. ---
try:
    import pycroscopy, sidpy
except Exception as exc:
    print('Installing / repairing pycroscopy stack because import failed:', repr(exc))
    !pip install -q --upgrade numpy scipy scikit-learn matplotlib h5py sidpy
    !pip install -q --upgrade --force-reinstall --no-deps pycroscopy
    import pycroscopy, sidpy

import io, contextlib, warnings
import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from pycroscopy.image import ImageWindowing
import sidpy

warnings.filterwarnings('ignore', module=r'sidpy.*')
plt.rcParams.update({'figure.dpi': 110, 'image.cmap': 'afmhot'})
rng = np.random.default_rng(7)
print('pycroscopy', pycroscopy.__version__, '| sidpy', sidpy.__version__)

## 1. Simulate the FeSeTe surface

FeSe$_x$Te$_{1-x}$ cleaves to expose a square chalcogen lattice. The paper studies FeSe$_{0.45}$Te$_{0.55}$ and finds clustering of like atoms. We mimic this with a binary lattice model where unlike Se-Te nearest-neighbor pairs cost an energy $w_0$ in units of $k_B T$.

The Monte Carlo moves swap Se and Te atoms, so the global composition stays fixed. Larger positive $w_0$ means stronger segregation.

In [ ]:
def neighbor_sum_energy(spins, w0=0.5):
    """Energy penalty w0 for unlike horizontal/vertical nearest-neighbor pairs."""
    return w0 * ((spins != np.roll(spins, 1, axis=0)).sum() +
                 (spins != np.roll(spins, 1, axis=1)).sum())


def make_random_lattice(n=48, x_se=0.45, rng=None):
    """Binary lattice: 1 = Se (bright), 0 = Te (dark)."""
    rng = np.random.default_rng() if rng is None else rng
    flat = np.zeros(n*n, dtype=np.int8)
    flat[:int(round(x_se*n*n))] = 1
    rng.shuffle(flat)
    return flat.reshape(n, n)


def local_pair_energy(spins, r, c, w0):
    val = spins[r, c]
    nbrs = [spins[(r-1) % spins.shape[0], c], spins[(r+1) % spins.shape[0], c],
            spins[r, (c-1) % spins.shape[1]], spins[r, (c+1) % spins.shape[1]]]
    return w0 * sum(val != nb for nb in nbrs)


def kawasaki_mc(n=48, x_se=0.45, w0=0.55, sweeps=500, burn=100, sample_every=25, rng=None):
    """Fixed-composition Metropolis MC for a binary lattice."""
    rng = np.random.default_rng() if rng is None else rng
    spins = make_random_lattice(n, x_se, rng)
    samples = []
    n_moves = n*n
    for sweep in range(sweeps):
        for _ in range(n_moves):
            r1, c1 = rng.integers(0, n, 2)
            r2, c2 = rng.integers(0, n, 2)
            if spins[r1, c1] == spins[r2, c2]:
                continue
            before = local_pair_energy(spins, r1, c1, w0) + local_pair_energy(spins, r2, c2, w0)
            spins[r1, c1], spins[r2, c2] = spins[r2, c2], spins[r1, c1]
            after = local_pair_energy(spins, r1, c1, w0) + local_pair_energy(spins, r2, c2, w0)
            dE = after - before
            if dE > 0 and rng.random() > np.exp(-dE):
                spins[r1, c1], spins[r2, c2] = spins[r2, c2], spins[r1, c1]
        if sweep >= burn and (sweep - burn) % sample_every == 0:
            samples.append(spins.copy())
    return spins, samples

n_atoms = 48
x_se = 0.45
true_w0 = 0.55
surface, mc_samples = kawasaki_mc(n=n_atoms, x_se=x_se, w0=true_w0, sweeps=450, burn=150,
                                  sample_every=30, rng=rng)
print(f'{len(mc_samples)} MC samples collected; final Se fraction = {surface.mean():.3f}')
print(f'Final unlike-pair energy / site = {neighbor_sum_energy(surface, true_w0) / surface.size:.3f} kBT')

fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
ax[0].imshow(surface, cmap='coolwarm', vmin=0, vmax=1, interpolation='nearest')
ax[0].set_title('simulated chalcogen lattice\n1 = Se, 0 = Te')
ax[1].imshow(gaussian_filter(surface.astype(float), 1.2), cmap='coolwarm', vmin=0, vmax=1)
ax[1].set_title('same lattice, visually smoothed')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 2. Render a simulated STM image

The real STM image in the paper contains bright and dark chalcogen sites. Here Se columns are brighter than Te columns, each site is rendered as a Gaussian peak, and the image includes background, blur, and Gaussian noise.

In [ ]:
def render_stm(spins, pitch=8, sigma=1.25, noise=0.12, background=0.12, rng=None):
    """Render a binary atom lattice as an STM-like image with known atom centers."""
    rng = np.random.default_rng() if rng is None else rng
    ny, nx = spins.shape
    h, w = ny*pitch, nx*pitch
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.zeros((h, w), dtype=float)
    centers = []
    for r in range(ny):
        for c in range(nx):
            y = r*pitch + pitch/2
            x = c*pitch + pitch/2
            amp = 1.25 if spins[r, c] else 0.70
            amp *= rng.normal(1.0, 0.04)
            img += amp * np.exp(-((xx-x)**2 + (yy-y)**2)/(2*sigma**2))
            centers.append((y, x))
    slow = background * (np.sin(2*np.pi*xx/w*1.1) + 0.7*np.cos(2*np.pi*yy/h*0.8))
    img = img + slow + rng.normal(0, noise, img.shape)
    img = img - img.min()
    img = img / img.max()
    return img, np.array(centers).reshape(ny, nx, 2)

pitch = 8
noise_sigma = 0.12
stm_image, centers = render_stm(surface, pitch=pitch, noise=noise_sigma, rng=rng)

fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
ax[0].imshow(stm_image, cmap='gray')
ax[0].set_title('simulated STM image')
ax[1].imshow(stm_image[:12*pitch, :12*pitch], cmap='gray')
ax[1].set_title('zoom: bright/dark atom sites')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 3. Physics codebook: the 12 nearest-neighbor configurations

For every central atom, we examine its four nearest neighbors: north, east, south, and west. There are $2^5 = 32$ raw binary patterns, but rotations and mirror symmetries of the square lattice reduce them to 12 distinct local configurations. These are the bins of the physics codebook.

In [ ]:
def transforms_4(v):
    """All rotations/reflections of a four-neighbor ring ordered N,E,S,W."""
    v = tuple(v)
    rots = [v[i:] + v[:i] for i in range(4)]
    refl = tuple([v[0], v[3], v[2], v[1]])
    rots += [refl[i:] + refl[:i] for i in range(4)]
    return rots


def canonical_config(center, neighbors):
    return (int(center), min(transforms_4(tuple(int(x) for x in neighbors))))

# Enumerate the complete symmetry-reduced nearest-neighbor codebook.
physics_codebook = []
for center in [0, 1]:
    reps = sorted({canonical_config(center, nbs) for nbs in itertools.product([0, 1], repeat=4)})
    physics_codebook.extend(reps)
code_to_index = {cfg: i for i, cfg in enumerate(physics_codebook)}
print(f'{len(physics_codebook)} symmetry-distinct configurations')
for i, cfg in enumerate(physics_codebook):
    print(f'{i:02d}: center={cfg[0]}, neighbors(N,E,S,W)={cfg[1]}')


def local_config_indices(spins):
    idx = []
    for r in range(spins.shape[0]):
        for c in range(spins.shape[1]):
            nbs = (spins[(r-1) % spins.shape[0], c], spins[r, (c+1) % spins.shape[1]],
                   spins[(r+1) % spins.shape[0], c], spins[r, (c-1) % spins.shape[1]])
            idx.append(code_to_index[canonical_config(spins[r, c], nbs)])
    return np.array(idx, dtype=int)


def config_histogram(spins, normalize=True):
    counts = np.bincount(local_config_indices(spins), minlength=len(physics_codebook)).astype(float)
    if normalize:
        counts /= counts.sum()
    return counts


def plot_codebook(codebook):
    fig, ax = plt.subplots(2, 6, figsize=(10, 3.8))
    for i, (center, nbs) in enumerate(codebook):
        a = ax.ravel()[i]
        patch = np.full((3, 3), np.nan)
        patch[1, 1] = center
        patch[0, 1], patch[1, 2], patch[2, 1], patch[1, 0] = nbs
        a.imshow(patch, cmap='coolwarm', vmin=0, vmax=1)
        a.set_title(str(i), fontsize=10)
        a.set_xticks([]); a.set_yticks([])
    plt.suptitle('physics codebook: 12 symmetry-distinct nearest-neighbor configurations')
    plt.tight_layout()

plot_codebook(physics_codebook)

In [ ]:
physics_hist = config_histogram(surface)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(np.arange(len(physics_hist)), physics_hist, color='steelblue')
ax.set_xlabel('configuration index')
ax.set_ylabel('relative frequency')
ax.set_title('target histogram from the known simulated lattice')
ax.set_xticks(np.arange(len(physics_hist)))
plt.tight_layout()

## 4. Statistical distance fit of the interaction energy

The paper compares experimental and simulated histograms using statistical distance,

$$s(p,q) = \arccos \left( \sum_i \sqrt{p_i q_i} \right),$$

where $p_i$ and $q_i$ are probabilities for local configuration bin $i$. This is closely related to the Bhattacharyya coefficient and treats the two histograms symmetrically.

For a workshop-speed demonstration, we build a small model library by simulating several trial values of $w_0$ and choose the one with minimum distance to the target histogram.

In [ ]:
def statistical_distance(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    bc = np.sqrt(p*q).sum()
    return np.arccos(np.clip(bc, 0, 1))


def simulate_histogram_at_w(w0, n=48, x_se=0.45, sweeps=260, burn=100, sample_every=40, seed=0):
    rr = np.random.default_rng(seed)
    final, samples = kawasaki_mc(n=n, x_se=x_se, w0=w0, sweeps=sweeps, burn=burn,
                                 sample_every=sample_every, rng=rr)
    hists = [config_histogram(s) for s in samples]
    if not hists:
        hists = [config_histogram(final)]
    return np.mean(hists, axis=0)

w_grid = np.linspace(0.0, 1.2, 17)
model_hists = []
for i, w in enumerate(w_grid):
    model_hists.append(simulate_histogram_at_w(w, n=n_atoms, x_se=x_se, seed=100+i))
model_hists = np.array(model_hists)

distances = np.array([statistical_distance(physics_hist, mh) for mh in model_hists])
best = distances.argmin()
est_w0 = w_grid[best]
print(f'true w0 = {true_w0:.2f} kBT')
print(f'estimated w0 from physics codebook = {est_w0:.2f} kBT')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(w_grid, distances**2, 'o-')
ax[0].axvline(true_w0, color='k', ls='--', label='true')
ax[0].axvline(est_w0, color='tab:red', ls=':', label='fit')
ax[0].set_xlabel('$w_0$ ($k_B T$)')
ax[0].set_ylabel('$s^2$')
ax[0].set_title('statistical-distance landscape')
ax[0].legend()

ax[1].bar(np.arange(12)-0.18, physics_hist, width=0.36, label='target')
ax[1].bar(np.arange(12)+0.18, model_hists[best], width=0.36, label='best model')
ax[1].set_xlabel('configuration index')
ax[1].set_ylabel('relative frequency')
ax[1].set_title('histogram match')
ax[1].legend()
plt.tight_layout()

## 5. Windowed k-means codebook using pycroscopy

The physics route assumed that the atomic species were already labeled. In a real image, we need to infer those labels. Here we use `pycroscopy.image.ImageWindowing` to extract one local image window per lattice site, then run k-means on the window intensities.

This is intentionally simple: it lets us see how an image-analysis uncertainty propagates into a physical parameter. More advanced workflows could add atom finding, drift correction, subpixel fitting, or Bayesian classification.

In [ ]:
def atom_windows_with_pycroscopy(image, pitch=8):
    """Extract one local window per lattice site using pycroscopy ImageWindowing."""
    ds = sidpy.Dataset.from_array(image, name='simulated_FeSeTe_STM')
    ds.data_type = 'image'
    parms = dict(window_size_x=pitch, window_size_y=pitch,
                 window_step_x=pitch, window_step_y=pitch,
                 mode='image')
    with contextlib.redirect_stdout(io.StringIO()):
        windows = np.array(ImageWindowing(parms).MakeWindows(ds))
    return windows

windows = atom_windows_with_pycroscopy(stm_image, pitch=pitch)
n_wx, n_wy = windows.shape[:2]
X_windows = windows.reshape(n_wx*n_wy, -1)
X_scaled = StandardScaler().fit_transform(X_windows)

km = KMeans(n_clusters=2, n_init=20, random_state=0)
labels_flat = km.fit_predict(X_scaled)

# ImageWindowing returns the grid as [x, y]. Transpose to match lattice display [row, col].
labels_grid = labels_flat.reshape(n_wx, n_wy).T

# Identify which cluster is bright by comparing raw mean intensity.
mean_intensity = [X_windows[labels_flat == lab].mean() for lab in range(2)]
se_cluster = int(np.argmax(mean_intensity))
kmeans_lattice = (labels_grid == se_cluster).astype(np.int8)

accuracy = max((kmeans_lattice == surface).mean(), (1 - kmeans_lattice == surface).mean())
print(f'Window grid from pycroscopy: {windows.shape}')
print(f'k-means label accuracy against the simulation truth: {accuracy:.3f}')

fig, ax = plt.subplots(1, 4, figsize=(12, 3.6))
ax[0].imshow(stm_image, cmap='gray'); ax[0].set_title('STM image')
ax[1].imshow(windows[n_wx//3, n_wy//2], cmap='gray'); ax[1].set_title('example window')
ax[2].imshow(kmeans_lattice, cmap='coolwarm', vmin=0, vmax=1, interpolation='nearest')
ax[2].set_title('k-means labels')
ax[3].imshow(surface, cmap='coolwarm', vmin=0, vmax=1, interpolation='nearest')
ax[3].set_title('ground truth')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

In [ ]:
kmeans_hist = config_histogram(kmeans_lattice)
kmeans_distances = np.array([statistical_distance(kmeans_hist, mh) for mh in model_hists])
k_best = kmeans_distances.argmin()
k_est_w0 = w_grid[k_best]
print(f'estimated w0 from windowed k-means codebook = {k_est_w0:.2f} kBT')
print(f'statistical distance between physics and k-means histograms = {statistical_distance(physics_hist, kmeans_hist):.4f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(np.arange(12)-0.18, physics_hist, width=0.36, label='physics labels')
ax[0].bar(np.arange(12)+0.18, kmeans_hist, width=0.36, label='windowed k-means')
ax[0].set_xlabel('configuration index')
ax[0].set_ylabel('relative frequency')
ax[0].set_title('two codebooks, same bins')
ax[0].legend()

ax[1].plot(w_grid, distances**2, 'o-', label='physics labels')
ax[1].plot(w_grid, kmeans_distances**2, 's-', label='windowed k-means')
ax[1].axvline(true_w0, color='k', ls='--', label='true')
ax[1].set_xlabel('$w_0$ ($k_B T$)')
ax[1].set_ylabel('$s^2$')
ax[1].set_title('fit comparison')
ax[1].legend()
plt.tight_layout()

## 6. Uncertainty from finite image area

The paper estimated error bars by dividing the image into 9 blocks. We can do the same. Each block gives a smaller histogram, which gives a spread of fitted interaction energies.

In [ ]:
def block_hists(spins, n_blocks=3):
    hists = []
    nr, nc = spins.shape
    br, bc = nr // n_blocks, nc // n_blocks
    for i in range(n_blocks):
        for j in range(n_blocks):
            block = spins[i*br:(i+1)*br, j*bc:(j+1)*bc]
            hists.append(config_histogram(block))
    return np.array(hists)


def fit_w_from_hist(hist, model_hists=model_hists, w_grid=w_grid):
    ds = np.array([statistical_distance(hist, mh) for mh in model_hists])
    return w_grid[ds.argmin()], ds.min()

block_estimates = np.array([fit_w_from_hist(h)[0] for h in block_hists(kmeans_lattice, 3)])
print('block-wise w0 estimates:', np.round(block_estimates, 2))
print(f'mean +/- std = {block_estimates.mean():.2f} +/- {block_estimates.std(ddof=1):.2f} kBT')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(block_estimates, bins=np.arange(-0.03, 1.26, 0.12), color='tab:purple', alpha=0.8)
ax.axvline(true_w0, color='k', ls='--', label='true')
ax.set_xlabel('$w_0$ estimate ($k_B T$)')
ax.set_ylabel('number of image blocks')
ax.set_title('finite-area uncertainty')
ax.legend()
plt.tight_layout()

## 7. What happens when the image gets noisier?

Now we repeat the render -> pycroscopy windowing -> k-means labels -> histogram -> fit pipeline for several image noise levels. The physical lattice is unchanged, so any broadening of $w_0$ estimates is caused by harder image classification.

In [ ]:
def estimate_from_noisy_image(spins, noise, repeats=6, pitch=8, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    estimates, accuracies = [], []
    for rep in range(repeats):
        img, _ = render_stm(spins, pitch=pitch, noise=noise, rng=rng)
        wins = atom_windows_with_pycroscopy(img, pitch=pitch)
        nxw, nyw = wins.shape[:2]
        X = wins.reshape(nxw*nyw, -1)
        labs = KMeans(n_clusters=2, n_init=10, random_state=rep).fit_predict(StandardScaler().fit_transform(X))
        grid = labs.reshape(nxw, nyw).T
        means = [X[labs == lab].mean() for lab in range(2)]
        pred = (grid == int(np.argmax(means))).astype(np.int8)
        accuracies.append(max((pred == spins).mean(), (1 - pred == spins).mean()))
        estimates.append(fit_w_from_hist(config_histogram(pred))[0])
    return np.array(estimates), np.array(accuracies)

noise_levels = [0.05, 0.12, 0.20, 0.30]
noise_results = {}
for ns in noise_levels:
    est, acc = estimate_from_noisy_image(surface, ns, repeats=6, pitch=pitch, rng=rng)
    noise_results[ns] = (est, acc)
    print(f'noise={ns:.2f}: w0={est.mean():.2f} +/- {est.std(ddof=1):.2f} kBT, accuracy={acc.mean():.3f}')

means = np.array([noise_results[ns][0].mean() for ns in noise_levels])
stds = np.array([noise_results[ns][0].std(ddof=1) for ns in noise_levels])
accs = np.array([noise_results[ns][1].mean() for ns in noise_levels])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].errorbar(noise_levels, means, yerr=stds, marker='o', capsize=4)
ax[0].axhline(true_w0, color='k', ls='--', label='true')
ax[0].set_xlabel('image noise sigma')
ax[0].set_ylabel('$w_0$ estimate ($k_B T$)')
ax[0].set_title('noise increases parameter uncertainty')
ax[0].legend()
ax[1].plot(noise_levels, accs, 'o-')
ax[1].set_xlabel('image noise sigma')
ax[1].set_ylabel('k-means label accuracy')
ax[1].set_ylim(0.5, 1.02)
ax[1].set_title('classification gets harder')
plt.tight_layout()

## Exercises

1. **Noise and uncertainty.** Increase `noise_levels` to include 0.40 and 0.50. At what point does the fitted $w_0$ become biased, not just uncertain?
2. **Image area.** Change `n_atoms` from 48 to 24 and then to 72. Re-run the notebook sections that generate data and fit $w_0$. How does the block-wise spread change?
3. **Weaker segregation.** Set `true_w0 = 0.20`. Are the physics and k-means codebooks still able to distinguish the model from a random solid solution?
4. **Codebook complexity.** Add next-nearest neighbors to the descriptor. How many symmetry-distinct bins do you get, and do they improve the fit or mostly add noise?
5. **Window size.** In `atom_windows_with_pycroscopy`, try `window_size_x = window_size_y = 6`, `8`, and `12` while keeping `window_step_x = window_step_y = pitch`. Which value best separates bright and dark atoms?
6. **K-means failure mode.** Add a stronger slowly varying background in `render_stm`. Does k-means start clustering by background instead of atom type? Try subtracting a blurred image as a background correction before windowing.
7. **Model mismatch.** Generate the target lattice with anisotropic interactions, e.g. a larger penalty for horizontal unlike pairs than vertical unlike pairs. Can the one-parameter model detect that it is wrong? What does the residual histogram tell you?
8. **Connection to the paper.** The paper fits $w_0$, $w_1$, and $w_{-1}$ and finds that $w_0$ is much better constrained by the surface image. Design a synthetic 3D extension of this notebook where hidden layers are sampled, then test when the interlayer terms become identifiable.